# Notebook 05 — Model Registry

**RHOAI 3.5 | MLflow Model Registry**

The MLflow Model Registry provides a central store for model versions with:
- Named models with semantic versioning
- **Aliases** (e.g. `champion`, `challenger`) instead of legacy stage names
- Metadata: description, tags, lineage back to the training run
- Easy loading via URI (`models:/<name>@<alias>` or `models:/<name>/<version>`)

This notebook demonstrates:
1. Registering a model from a training run
2. Creating model versions
3. Setting aliases (`champion` / `challenger`)
4. Loading models from the registry
5. Promoting challenger → champion
6. Searching the registry

---
**Prerequisite:** Run notebooks 02 or 03 first to populate runs with logged models.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from mlflow.sklearn import SERIALIZATION_FORMAT_CLOUDPICKLE
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

client = MlflowClient()
EXPERIMENT_NAME  = "rhoai-mlflow-demo"
REGISTERED_MODEL = "wine-quality-classifier"

mlflow.set_experiment(EXPERIMENT_NAME)

# Prepare data
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Registered model name : {REGISTERED_MODEL}")
print(f"Tracking URI          : {mlflow.get_tracking_uri()}")

## Section 1 — Create a Registered Model

In [ ]:
# Create the registered model entry (idempotent — safe to re-run)
try:
    registered_model = client.create_registered_model(
        name=REGISTERED_MODEL,
        description="Wine variety classification model trained on the UCI Wine dataset. "
                    "Input: 13 physicochemical features. Output: class 0, 1, or 2.",
        tags={
            "team"         : "data-science",
            "project"      : "wine-quality",
            "rhoai_version": "3.5",
            "framework"    : "scikit-learn",
        },
    )
    print(f"Created registered model: {registered_model.name}")
except mlflow.exceptions.MlflowException as e:
    if "already exists" in str(e).lower():
        print(f"Registered model '{REGISTERED_MODEL}' already exists — using existing.")
    else:
        raise

# Fetch and display current model info
rm = client.get_registered_model(REGISTERED_MODEL)
print(f"\nModel name       : {rm.name}")
print(f"Description      : {rm.description}")
print(f"Latest versions  : {[f'v{mv.version}({mv.current_stage})' for mv in rm.latest_versions]}")
print(f"Tags             : {rm.tags}")

## Section 2 — Register Version 1 (Random Forest)

In [ ]:
def train_rf(n_estimators=100, max_depth=5):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    model.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_s))
    return model, acc

rf_model, rf_acc = train_rf(n_estimators=100, max_depth=5)
signature = infer_signature(X_train_s, rf_model.predict(X_train_s))

with mlflow.start_run(run_name="registry-rf-v1") as run:
    mlflow.log_params({"model_type": "RandomForest", "n_estimators": 100, "max_depth": 5})
    mlflow.log_metric("test_accuracy", rf_acc)
    mlflow.set_tags({"notebook": "05_model_registry", "version_candidate": "v1"})

    model_info = mlflow.sklearn.log_model(
        rf_model,
        name="model",
        serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
        signature=signature,
        input_example=X_test_s[:3],
        registered_model_name=REGISTERED_MODEL,
    )
    v1_run_id = run.info.run_id

print(f"Registered model URI  : {model_info.model_uri}")
print(f"Test accuracy         : {rf_acc:.4f}")

versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
v1 = sorted(versions, key=lambda v: int(v.version))[-1]
v1_version = v1.version

client.update_model_version(
    name=REGISTERED_MODEL,
    version=v1_version,
    description=f"Random Forest, n_estimators=100, max_depth=5. Test accuracy: {rf_acc:.4f}.",
)
client.set_model_version_tag(REGISTERED_MODEL, v1_version, "algorithm",    "RandomForest")
client.set_model_version_tag(REGISTERED_MODEL, v1_version, "test_accuracy", f"{rf_acc:.4f}")

print(f"Version {v1_version} created and tagged.")

## Section 3 — Set the `champion` Alias

Aliases replace legacy stage names (`Staging`, `Production`). They are:
- Mutable: you can point them to any version at any time
- Human-readable: `models:/wine-quality-classifier@champion`
- Flexible: one model can have multiple aliases

In [ ]:
# Promote version 1 to champion
client.set_registered_model_alias(
    name=REGISTERED_MODEL,
    alias="champion",
    version=v1_version,
)
print(f"Alias 'champion' → version {v1_version}")

# Verify
champion = client.get_model_version_by_alias(REGISTERED_MODEL, "champion")
print(f"champion version : {champion.version}")
print(f"champion run ID  : {champion.run_id}")
print(f"champion source  : {champion.source}")

## Section 4 — Register Version 2 (Gradient Boosting) as `challenger`

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
gb_model.fit(X_train_s, y_train)
gb_acc = accuracy_score(y_test, gb_model.predict(X_test_s))

signature_gb = infer_signature(X_train_s, gb_model.predict(X_train_s))

with mlflow.start_run(run_name="registry-gb-v2") as run:
    mlflow.log_params({
        "model_type"    : "GradientBoosting",
        "n_estimators"  : 200,
        "learning_rate" : 0.05,
        "max_depth"     : 3,
    })
    mlflow.log_metric("test_accuracy", gb_acc)
    mlflow.set_tags({"notebook": "05_model_registry", "version_candidate": "v2"})

    model_info_gb = mlflow.sklearn.log_model(
        gb_model,
        name="model",
        serialization_format=SERIALIZATION_FORMAT_CLOUDPICKLE,
        signature=signature_gb,
        input_example=X_test_s[:3],
        registered_model_name=REGISTERED_MODEL,
    )
    v2_run_id = run.info.run_id

versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
v2 = sorted(versions, key=lambda v: int(v.version))[-1]
v2_version = v2.version

client.update_model_version(
    name=REGISTERED_MODEL,
    version=v2_version,
    description=f"GradientBoosting, n_estimators=200, lr=0.05. Test accuracy: {gb_acc:.4f}.",
)
client.set_model_version_tag(REGISTERED_MODEL, v2_version, "algorithm",    "GradientBoosting")
client.set_model_version_tag(REGISTERED_MODEL, v2_version, "test_accuracy", f"{gb_acc:.4f}")

client.set_registered_model_alias(REGISTERED_MODEL, "challenger", v2_version)

print(f"Version {v2_version} registered as 'challenger'.")
print(f"  GradientBoosting test accuracy: {gb_acc:.4f}")
print(f"  RandomForest test accuracy    : {rf_acc:.4f}")

## Section 5 — Load Models from the Registry

URI formats:
- By alias: `models:/<name>@<alias>`
- By version: `models:/<name>/<version>`
- Latest: `models:/<name>/latest` (deprecated — use aliases instead)

In [ ]:
# Load the champion model by alias
champion_uri = f"models:/{REGISTERED_MODEL}@champion"
champion_model = mlflow.sklearn.load_model(champion_uri)

preds = champion_model.predict(X_test_s)
acc   = accuracy_score(y_test, preds)

print(f"Loaded model URI  : {champion_uri}")
print(f"Model type        : {type(champion_model).__name__}")
print(f"Test accuracy     : {acc:.4f}")
print(f"Sample predictions: {preds[:10]}, actual: {y_test[:10]}")

In [ ]:
# Load challenger and compare
challenger_uri   = f"models:/{REGISTERED_MODEL}@challenger"
challenger_model = mlflow.sklearn.load_model(challenger_uri)

challenger_acc = accuracy_score(y_test, challenger_model.predict(X_test_s))

print(f"Champion  ({type(champion_model).__name__:30s}): acc={acc:.4f}")
print(f"Challenger({type(challenger_model).__name__:30s}): acc={challenger_acc:.4f}")
print()
if challenger_acc > acc:
    print("Challenger outperforms champion — promoting in the next cell.")
else:
    print("Champion holds its ground — no promotion needed.")

## Section 6 — Promote Challenger to Champion

In [ ]:
if challenger_acc > acc:
    # Move 'champion' alias to the challenger version
    client.set_registered_model_alias(REGISTERED_MODEL, "champion",    v2_version)
    # Keep old champion accessible as 'previous-champion'
    client.set_registered_model_alias(REGISTERED_MODEL, "previous-champion", v1_version)
    # Remove 'challenger' alias from v2 since it's now champion
    client.delete_registered_model_alias(REGISTERED_MODEL, "challenger")

    print(f"champion         → version {v2_version} (GradientBoosting, acc={challenger_acc:.4f})")
    print(f"previous-champion → version {v1_version} (RandomForest, acc={acc:.4f})")
else:
    print(f"No promotion: champion stays at version {v1_version}.")
    # Clean up challenger alias
    client.delete_registered_model_alias(REGISTERED_MODEL, "challenger")
    print("challenger alias removed.")

# Final state
rm = client.get_registered_model(REGISTERED_MODEL)
print()
print(f"Final versions in '{REGISTERED_MODEL}':")
for mv in sorted(client.search_model_versions(f"name='{REGISTERED_MODEL}'"), key=lambda v: int(v.version)):
    aliases = client.get_model_version_by_alias
    print(f"  v{mv.version} | {mv.tags.get('algorithm', 'unknown'):20s} | acc={mv.tags.get('test_accuracy', 'n/a')} | status={mv.status}")

## Section 7 — Search the Registry

In [ ]:
# Search all registered models in the workspace
all_models = client.search_registered_models()
print(f"Registered models in workspace: {len(all_models)}")
for rm in all_models:
    latest = [f"v{mv.version}" for mv in rm.latest_versions]
    print(f"  {rm.name:40s} | versions: {latest}")

print()

# Search model versions by tag
print(f"Versions of '{REGISTERED_MODEL}' with GradientBoosting algorithm:")
gb_versions = client.search_model_versions(
    filter_string=f"name='{REGISTERED_MODEL}' and tags.algorithm='GradientBoosting'"
)
for mv in gb_versions:
    print(f"  v{mv.version} | acc={mv.tags.get('test_accuracy')} | run_id={mv.run_id[:12]}...")

In [ ]:
# Load as generic pyfunc for framework-agnostic serving
champion_pyfunc = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@champion")

# Create a proper DataFrame with feature names (pyfunc may require this)
sample_df = pd.DataFrame(X_test_s[:5], columns=wine.feature_names)
predictions = champion_pyfunc.predict(sample_df)

print("pyfunc predictions (champion model):")
for i, (pred, actual) in enumerate(zip(predictions, y_test[:5])):
    status = "✓" if pred == actual else "✗"
    print(f"  Sample {i+1}: predicted={wine.target_names[int(pred)]:8s}  actual={wine.target_names[actual]:8s}  {status}")

## Summary

| Operation | API |
|-----------|-----|
| Create registered model | `client.create_registered_model(name, description, tags)` |
| Register during training | `mlflow.sklearn.log_model(..., registered_model_name=...)` |
| Add description / tag | `client.update_model_version()`, `client.set_model_version_tag()` |
| Set alias | `client.set_registered_model_alias(name, alias, version)` |
| Load by alias | `mlflow.sklearn.load_model("models:/<name>@<alias>")` |
| Load by version | `mlflow.sklearn.load_model("models:/<name>/<version>")` |
| Load as pyfunc | `mlflow.pyfunc.load_model(uri)` |
| Search models | `client.search_registered_models()` |
| Search versions | `client.search_model_versions(filter_string)` |
| Remove alias | `client.delete_registered_model_alias(name, alias)` |

---

## All Notebooks Complete

| Notebook | Status |
|----------|--------|
| 01 — Setup & Authentication | ✓ |
| 02 — Experiment Tracking | ✓ |
| 03 — Hyperparameter Tuning & Run Comparison | ✓ |
| 04 — GenAI Tracing | ✓ |
| 05 — Model Registry | ✓ |

**RHOAI Dashboard:** `Develop & train > Experiments (MLflow)` to explore all runs, traces, and registered models.